### `1.字符串`

Python 规定所有的对象都应该生成两个不同的字符串表示：
- 一种是人类可读的文本，另一种是 Python 可解释的表示式。字符串的构造函数，即 str，返回一个人类可读的字符串。
- 如果可能，repr 函数返回一个 Python 可解释的表达式，该表达式的求值结果与原对象相同。

In [1]:
12e12

12000000000000.0

In [2]:
print(repr(12e12))

12000000000000.0


In [3]:
repr(min)

'<built-in function min>'

In [5]:
"""
    str 构造器通常与 repr 行为一致，但是在某些情况下它会提供一个更容易解释的文本表示。
    例如，我们可以看到 str 和 repr 对于日期对象的不同展示。
"""
from datetime import date
tues = date(2026, 9, 3)
print(repr(tues))
print(str(tues))

datetime.date(2026, 9, 3)
2026-09-03


In [6]:
tues.__repr__()

'datetime.date(2026, 9, 3)'

In [7]:
tues.__str__()

'2026-09-03'

### `2.专用方法`

在 Python 中，某些特殊名称会在特殊情况下被 Python 解释器调用。例如，类的 `__init__ `方法会在对象被创建时自动调用。`__str__ `方法会在打印时自动调用，`__repr__` 方法会在交互式环境显示其值的时候自动调用。

In [8]:
# bool值 __bool__方法

In [9]:
# 序列操作中的len函数
len("Go bears!")

9

In [10]:
"Go bears!".__len__()

9

In [11]:
# __getitem__方法
print("Go bears!"[3])
print("Go bears!".__getitem__(3))

b
b


In [12]:
# 可调用对象
def make_adder(n):
    def adder(k):
        return k + n
    return adder

adder = make_adder(5)
print(adder(2))

7


In [13]:
class Adder(object):
    def __init__(self, n):
        self.n = n
    
    def __call__(self, k):
        return k + self.n

add_three_object = Adder(3)
print(add_three_object(4))

7


### `3.多重表示`

In [14]:
class Numer:
    def __add__(self, other):
        return self.add(other)
    
    def __mul__(self, other):
        return self.mul(other)

In [15]:
class Complex(Numer):

    def add(self, other):
        return ComplexRI(self.real + other.real, self.imag + other.imag)
    
    def mul(self, other):
        magnitude = self.magnitude * other.magnitude
        return ComplexMA(magnitude, self.angle + other.angle)

In [16]:
# @property 修饰符允许函数在没有调用表达式语法（表达式后跟随圆括号）的情况下被调用
from math import atan2
class ComplexRI(Complex):
    def __init__(self, real, imag):
        self.real = real
        self.imag = imag

    @property
    def magnitude(self):
        return (self.real ** 2 + self.imag ** 2) ** 0.5

    @property
    def angle(self):
        return atan2(self.imag, self.real)
    
    def __repr__(self):
        return "ComplexRI({}, {})".format(self.real, self.imag)

In [ ]:
ri = ComplexRI(5, 12)
ri.real # 5
ri.magnitude # 13.0

ri.real = 9
ri.real # 9
ri.magnitude # 15.0

15.0

In [21]:
from math import cos, sin, pi
class ComplexMA(Complex):
    def __init__(self, magnitude, angle):
        self.magnitude = magnitude
        self.angle = angle
    
    @property
    def real(self):
        return self.magnitude * cos(self.angle)
    
    @property
    def imag(self):
        return self.magnitude * sin(self.angle)
    
    def __repr__(self):
        return "ComplexMA({}, {})".format(self.magnitude, self.angle)

In [22]:
ma = ComplexMA(2, pi / 2)
ma.imag

2.0

In [23]:
from math import pi
ComplexRI(3, 4) + ComplexMA(5, pi / 6)

ComplexRI(7.330127018922194, 6.5)

In [24]:
ComplexRI(0, 1) * ComplexRI(0, 1)

ComplexMA(1.0, 3.141592653589793)

### `4.泛型函数`
 

In [26]:
from math import gcd
class Rational(Numer):
    def __init__(self, numer, denom):
        g = gcd(numer, denom)
        self.numer = numer // g
        self.denom = denom // g
    
    def add(self, other):
        new_numer = self.numer * other.denom + self.denom * other.numer
        new_denom = self.denom * other.denom
        return Rational(new_numer, new_denom)
    
    def mul(self, other):
        return Rational(self.numer * other.numer, self.denom * other.denom)
    
    def __repr__(self):
        return "Rational({}, {})".format(self.numer, self.denom)

In [28]:
Rational(2, 5) + Rational(1, 10)

Rational(1, 2)

In [30]:
Rational(1, 4) * Rational(2, 3)

Rational(1, 6)

In [34]:
# 内置的函数 isinstance 接受一个对象或一个类。如果对象的类是所给的类或者继承自所给的类，它会返回一个真值。
c = ComplexRI(1, 1)
isinstance(c, ComplexRI) # True
isinstance(c, Complex) # True
isinstance(c, ComplexMA) # False

False

In [37]:
def is_real(c):
    if isinstance(c, ComplexRI):
        return c.imag == 0
    if isinstance(c, ComplexMA):
        return c.angle * pi == 0

In [38]:
is_real(ComplexRI(3, 0)) # True

True

In [40]:
Rational.type_tag = 'rat'
Complex.type_tag = 'com'
Rational(2, 5).type_tag == Rational(1, 2).type_tag # True
ComplexRI(1, 1).type_tag == ComplexMA(2, pi/2).type_tag # True
Rational(2, 5).type_tag == ComplexRI(1, 1).type_tag # False

False

In [41]:
def add_complex_and_rational(c, r):
    return ComplexRI(c.real + r.numer/r.denom, c.imag)

In [42]:
def mul_complex_and_rational(c, r):
    r_magnitude, r_angle = r.numer/r.denom, 0
    if r_magnitude < 0:
        r_magnitude, r_angle = -r_magnitude, pi
    return ComplexMA(c.magnitude * r_magnitude, c.angle + r_angle)

In [45]:
def add_rational_and_complex(r, c):
    return add_complex_and_rational(c, r)
def mul_rational_and_complex(r, c):
    return mul_complex_and_rational(c, r)

In [46]:
class Number:
    def __add__(self, other):
        if self.type_tag == other.type_tag:
            return self.add(other)
        elif (self.type_tag, other.type_tag) in self.adders:
            return self.cross_apply(other, self.adders)
    def __mul__(self, other):
        if self.type_tag == other.type_tag:
            return self.mul(other)
        elif (self.type_tag, other.type_tag) in self.multipliers:
            return self.cross_apply(other, self.multipliers)
    def cross_apply(self, other, cross_fns):
        cross_fn = cross_fns[(self.type_tag, other.type_tag)]
        return cross_fn(self, other)
    adders = {("com", "rat"): add_complex_and_rational,
                ("rat", "com"): add_rational_and_complex}
    multipliers = {("com", "rat"): mul_complex_and_rational,
                    ("rat", "com"): mul_rational_and_complex}

In [47]:
class Complex(Number):

    def add(self, other):
        return ComplexRI(self.real + other.real, self.imag + other.imag)
    
    def mul(self, other):
        magnitude = self.magnitude * other.magnitude
        return ComplexMA(magnitude, self.angle + other.angle)

In [48]:
# @property 修饰符允许函数在没有调用表达式语法（表达式后跟随圆括号）的情况下被调用
from math import atan2
class ComplexRI(Complex):
    def __init__(self, real, imag):
        self.real = real
        self.imag = imag

    @property
    def magnitude(self):
        return (self.real ** 2 + self.imag ** 2) ** 0.5

    @property
    def angle(self):
        return atan2(self.imag, self.real)
    
    def __repr__(self):
        return "ComplexRI({}, {})".format(self.real, self.imag)

In [49]:
from math import cos, sin, pi
class ComplexMA(Complex):
    def __init__(self, magnitude, angle):
        self.magnitude = magnitude
        self.angle = angle
    
    @property
    def real(self):
        return self.magnitude * cos(self.angle)
    
    @property
    def imag(self):
        return self.magnitude * sin(self.angle)
    
    def __repr__(self):
        return "ComplexMA({}, {})".format(self.magnitude, self.angle)

In [50]:
from math import gcd
class Rational(Number):
    def __init__(self, numer, denom):
        g = gcd(numer, denom)
        self.numer = numer // g
        self.denom = denom // g
    
    def add(self, other):
        new_numer = self.numer * other.denom + self.denom * other.numer
        new_denom = self.denom * other.denom
        return Rational(new_numer, new_denom)
    
    def mul(self, other):
        return Rational(self.numer * other.numer, self.denom * other.denom)
    
    def __repr__(self):
        return "Rational({}, {})".format(self.numer, self.denom)

In [52]:
Complex.type_tag = 'com'
Rational.type_tag = 'rat'
ComplexRI(3, 4) + Rational(1, 2)

ComplexRI(3.5, 4)

In [53]:
# 强制转换
def rational_to_complex(r):
    return ComplexRI(r.numer/r.denom, 0)